# BIL545 · 3. Hafta · Colab defteri: veri ön işleme

Veri Madenciliğinde İleri Konular · Dr. Selma Güzel · Üsküdar Üniversitesi

Bu defterde, sunumdaki sırayla: veriyi okumak, eksik değerleri **işaretlemek**, künye tablosu, **doldurma** (medyan ve gösterge sütunu), **kimlik sütunu ve yinelenen kayıt** kontrolü, **ölçekleme ve uzaklık**, **ayrıklaştırma ve log dönüşümü** ve son olarak bir **sızıntı deneyi**.

> 2. haftadan devralınan bölümler (veriyi okumak, künye, uzaklık) burada da satır satır açıklanmıştır; defter tek başına çalışır.

**Bu defter nasıl kullanılır?**

* Hücreleri **yukarıdan aşağıya, sırayla** çalıştırın: hücreye tıklayıp **Shift + Enter**. Sonraki hücreler öncekilerde tanımlanan değişkenleri kullanır; ortadan başlarsanız `NameError` hatası alırsınız. Defteri baştan çalıştırmak için: Çalışma zamanı → Tümünü çalıştır.
* Sunumdaki her kod, burada **satır satır ya da birkaç satırlık parçalar hâlinde** verilmiştir. Her parçanın önünde iki açıklama vardır:
  * 🐍 **Python:** satırdaki Python kavramı (Python bilmeniz gerekmez).
  * 📊 **Konu:** satırın veri madenciliği açısından ne yaptığı ve neden yapıldığı.
* 🔍 **Ara adım** hücreleri sunumda yoktur: bir hesabın neye uygulandığını görebilmeniz için ara sonucu gösterir (ör. ortalamanın hangi sütunun hangi değerlerine uygulandığı).
* Her bölümün sonunda **sunumdaki hâli** tek hücre olarak tekrar verilir; çıktısı sunumdakiyle aynı olmalıdır. ✅ **Sonucu okuyalım** kutusu sonucu yorumlar.
* Kodda `#` işaretinden sonraki her şey **yorumdur**: Python onu çalıştırmaz, insanlar için yazılmış nottur.

**0. Sürümler**

🐍 **Python:** İki modülü açar ve sürümlerini yazar.

In [ ]:
import sklearn, pandas
print("scikit-learn:", sklearn.__version__)
print("pandas      :", pandas.__version__)

## 1. Veriyi okumak ve saklama tiplerine bakmak

**Konu.** Analizin ilk adımı veriyi bir **tabloya** (pandas'ta *DataFrame*) okumak ve neye benzediğine bakmaktır: kaç satır (kayıt), kaç sütun (öznitelik) var, her sütun bilgisayarda hangi tiple saklanıyor?

**Veri: diabetes (Pima Indians Diabetes).** 768 kadın hasta; 8 ölçüm ve hedef sütunu `class` (diyabet testi pozitif mi, negatif mi). Sütunlar: `preg` gebelik sayısı, `plas` glikoz, `pres` kan basıncı, `skin` deri kıvrım kalınlığı, `insu` insülin, `mass` beden kitle indeksi, `pedi` diyabet soy ağacı fonksiyonu, `age` yaş. Weka'da açtığımız `diabetes.arff` ile aynı veridir; burada OpenML adlı açık veri deposundan indiriyoruz.

🐍 **Python:** pandas modülünü açar ve ona `pd` takma adını verir. Bundan sonra pandas'ın her aracı `pd.` ile başlar (ör. `pd.DataFrame`).

📊 **Konu:** pandas, veriyi satır ve sütunlardan oluşan bir tablo olarak tutar ve işler; Excel tablosunun Python'daki karşılığı gibi düşünebilirsiniz.

In [ ]:
import pandas as pd

🐍 **Python:** scikit-learn'ün `datasets` (veri kümeleri) bölümünden yalnız `fetch_openml` işlevini açar.

📊 **Konu:** `fetch_openml`, OpenML'deki bir veri kümesini adıyla internetten indirir.

In [ ]:
from sklearn.datasets import fetch_openml

🐍 **Python:** İlk iki satır `#` ile başladığı için **yorumdur**, çalışmaz. Kendi CSV dosyanızı okumak isterseniz baştaki `# ` işaretini silip dosya adını yazarsınız (dosyayı önce Colab'ın sol taraftaki Dosyalar bölmesine yüklemeniz gerekir).

Üçüncü satır **zincirleme** okunur: `fetch_openml(...)` bir sonuç nesnesi döndürür; onun `.frame` özelliği, verinin tablo hâlidir. Bu tablo `df` adına atanır (*data frame*'in kısaltması; ders boyunca tabloya hep bu adı vereceğiz).

📊 **Konu:** `version=1`: bu adla birden çok sürüm olabilir; hangisini istediğimizi sabitliyoruz (tekrarlanabilirlik). `as_frame=True`: sonucu pandas tablosu olarak ver.

In [ ]:
# Veriniz bir CSV dosyası olsaydı böyle okunurdu:
# df = pd.read_csv("veri.csv")
df = fetch_openml("diabetes", version=1, as_frame=True).frame

🐍 **Python:** `.shape` tablonun boyutudur: (satır sayısı, sütun sayısı). Parantezsiz yazılır, çünkü bir işlev değil, tablonun bir **özelliğidir**.

📊 **Konu:** 768 kayıt, 9 sütun (8 öznitelik ve hedef `class`).

In [ ]:
print(df.shape)

🐍 **Python:** `.dtypes` her sütunun **saklama tipini** listeler. `float64`: ondalıklı sayı; `int64`: tam sayı; `category`: sınırlı sayıda değeri olan kategorik sütun; `object`: genellikle metin.

📊 **Konu:** Dikkat: saklama tipi, 2. haftada gördüğümüz **ölçek tipini** (nominal, sıralı, aralık, oran) söylemez. `float64` bir sütun oran ölçekli bir ölçüm de olabilir, 0 ile kodlanmış eksik değerler içeren bir ölçüm de. Weka'daki *Type* alanıyla aynı sınır: tipi siz belirlersiniz.

In [ ]:
print(df.dtypes)

🐍 **Python:** `.head()` tablonun ilk 5 satırını gösterir (`df.head(10)` ilk 10'u). Bir hücrenin **son satırı** bir değer ise Colab onu `print` olmadan da güzel bir tablo olarak gösterir.

📊 **Konu:** İlk satırlarda `skin` ve `insu` sütunlarında 0 değerlerini görün: 0 deri kalınlığı ya da 0 insülin biyolojik olarak mümkün değildir. Bunlar ölçülmemiş, yani **eksik** değerlerdir.

In [ ]:
df.head()

✅ **Sonucu okuyalım:** Tablo 768 × 9. Sekiz ölçümün tamamı `float64`, `class` ise `category` (iki değer: `tested_negative`, `tested_positive`). pandas "eksik yok" der, çünkü eksik değerler boş bırakılmamış, **0** yazılmış.

## 2. Eksik değerle ilk iş: işaretlemek

**Konu.** diabetes verisinde ölçülmemiş değerler **0** yazılmış: glikoz (`plas`), kan basıncı (`pres`), deri kalınlığı (`skin`), insülin (`insu`) ve beden kitle indeksi (`mass`) sıfır olamaz. Araçlar bu sıfırları gerçek ölçüm sanar: ortalamaya katar, modele verir. Doldurmaya, silmeye ya da başka bir karara geçmeden önce aracın eksikliği **görmesi** gerekir. Bu yüzden ilk iş, sıfırları pandas'ın eksik değer işaretine, **NaN**'a (*Not a Number*, "sayı değil") çevirmektir. Weka'daki karşılığı `?`.

`preg` (gebelik sayısı) bu listede **yok**: 0 gebelik gerçek bir değerdir. Bu ayrımı veri değil, alan bilgisi yapar.

🐍 **Python:** numpy modülünü `np` takma adıyla açar. `np.nan` numpy'nin eksik değer işaretidir.

In [ ]:
import numpy as np

🐍 **Python:** Köşeli parantez içinde, virgülle ayrılmış öğeler bir **liste**dir. Burada beş sütun adını (metin) tutuyor. Python'da yerleşik bir "dizi" (array) tipi yoktur; liste her tür öğeyi tutabilir. Sayısal hesap için numpy'nin dizisi kullanılır.

📊 **Konu:** Sıfırın **eksik** anlamına geldiği sütunlar.

In [ ]:
sutunlar = ["plas", "pres", "skin", "insu", "mass"]   # preg YOK: orada 0 gerçek

🔍 **Ara adım:** İşaretlemeden önce her sütunda kaç sıfır var? `df[sutunlar]` tablodan yalnız bu beş sütunu seçer; `== 0` her hücreyi sıfırla karşılaştırır; `.sum()` her **sütundaki** True'ları sayar. `insu`'da 374, `skin`'de 227 sıfır.

In [ ]:
print((df[sutunlar] == 0).sum())

🐍 **Python:** Sağ taraf önce çalışır: `df[sutunlar]` beş sütunu seçer, `.replace(0, np.nan)` içlerindeki her 0'ı NaN ile değiştirip **yeni** bir tablo döndürür. Soldaki `df[sutunlar] = ...` bu sonucu `df`'in aynı sütunlarına **geri yazar**. Geri yazmazsak `df` değişmez.

📊 **Konu:** Artık bu sütunlardaki sıfırlar eksik olarak işaretli. `preg`'e dokunulmadı.

In [ ]:
df[sutunlar] = df[sutunlar].replace(0, np.nan)

🐍 **Python:** **Zincirleme çağrı**, soldan sağa okunur: `df.isna()` her hücre için "eksik mi?" (True/False tablosu), `.mean()` her **sütunun** True oranı, `.round(3)` üç basamak.

📊 **Konu:** `mean` burada ölçüm değerlerine değil, "eksik mi?" sorusunun True/False cevaplarına uygulanıyor; sonuç her sütunun **eksik oranıdır**.

In [ ]:
print(df.isna().mean().round(3))                     # şimdi eksik oranları görünür

✅ **Sonucu okuyalım:** `insu` %48,7, `skin` %29,6, `pres` %4,6, `mass` %1,4, `plas` %0,7 eksik. 2. haftada pandas "eksik yok" diyordu; aynı veri, işaretlendikten sonra gerçeği gösteriyor. Başka verilerde eksiklik −1, 99, 999, boş metin, "bilinmiyor" ya da "N/A" diye kodlanmış olabilir: veri sözlüğüne bakın, şüpheli ölçüde sık görünen değerleri kontrol edin.

## 3. İlk künye tablosu

**Konu.** **Veri künyesi**, her öznitelik için temel kontrolleri tek tabloda toplar: saklama tipi, eksik değer oranı, farklı değer sayısı, sıfır oranı ve kimlik şüphesi. Weka'da her özniteliğe tek tek tıklayarak baktığımız bilgileri (Type, Missing, Distinct) tek hücrede verir ve Weka'da olmayan iki kontrol ekler. Proje 1'in 1. bölümündeki tip ve eksiklik tablosunun başlangıcıdır.

Bu bölümde birçok hesap "bir tablo üzerinde, **sütun sütun**" yapılıyor. Her adımda hesabın **neye** uygulandığını ara adımlarla göreceğiz.

> Bu bölüm, sıfırları NaN olarak işaretledikten **sonra** çalışıyor. 2. haftadaki künyeyle karşılaştırın: eksik oranları artık görünür, sıfır oranları düştü.

🐍 **Python:** `select_dtypes("number")` tablodan yalnız **sayısal** sütunları seçer ve yeni bir tablo döndürür. Sonuç `sayisal` adına atanır; `df` değişmez.

📊 **Konu:** Sıfır oranı yalnız sayısal sütunlar için anlamlıdır. `class` kategorik olduğu için bu tabloya girmez.

In [ ]:
sayisal = df.select_dtypes("number")

🔍 **Ara adım:** `.columns` sütun adlarıdır; `.tolist()` onları düz bir listeye çevirir. İkinci listede `class` olmadığını görün.

In [ ]:
print(df.columns.tolist())
print(sayisal.columns.tolist())

Künyenin her sütununu ayrı ayrı hesaplayalım. Her hesap **bir sütun adı → bir sayı** eşlemesi üretir (pandas'ta buna *Series* denir: etiketli bir sayı listesi).

#### a) Saklama tipi

🔍 **Ara adım:** `df.dtypes` her sütunun tipini verir; `.astype(str)` bunları düz metne çevirir (tabloda düzgün görünsün diye).

In [ ]:
df.dtypes.astype(str)

#### b) Eksik oranı: `df.isna().mean()`

🔍 **Ara adım:** **1. adım:** `df.isna()` tablonun **her hücresi** için "bu hücre eksik (NaN) mi?" sorusunu sorar ve aynı boyutta bir **True/False tablosu** döndürür. İlk 5 satırını görüyoruz. `insu` ve `skin` sütunlarında True'lar var: sıfırları NaN yapmıştık.

In [ ]:
df.isna().head()

🔍 **Ara adım:** **2. adım (anlamak için):** `.sum()` her **sütundaki** True'ları sayar (True = 1, False = 0). Sonuç: her sütunda kaç eksik hücre var.

In [ ]:
df.isna().sum()

🔍 **Ara adım:** **3. adım:** `.mean()` her **sütunun** ortalamasını alır. True/False'ların ortalaması, True'ların **oranıdır**: 768 hücreden kaçının eksik olduğu ÷ 768. `.round(3)` üç basamağa yuvarlar. Yani `mean` burada tablonun değerlerine değil, **eksik mi? sorusunun cevaplarına** uygulanıyor.

In [ ]:
df.isna().mean().round(3)

#### c) Farklı değer sayısı: `df.nunique()`

🔍 **Ara adım:** `nunique()` (*number of unique*) her sütunda **kaç farklı değer** olduğunu sayar. Weka'daki *Distinct*. `class` için 2, `preg` için 17. Sürekli ölçümlerde (ör. `pedi`) bu sayı yüksektir.

In [ ]:
df.nunique()

#### d) Sıfır oranı: `(sayisal == 0).mean()`

🔍 **Ara adım:** **1. adım:** `sayisal == 0` her hücreyi 0 ile karşılaştırır; yine bir True/False tablosu (yalnız sayısal sütunlar).

In [ ]:
(sayisal == 0).head()

🔍 **Ara adım:** **2. adım:** her sütunun True oranı = o sütunda **0 olan değerlerin oranı**. Eksik oranıyla aynı mantık: `mean` sayıların kendisine değil, "sıfır mı?" cevaplarına uygulanıyor.

In [ ]:
(sayisal == 0).mean().round(3)

#### e) Hepsini bir tabloda birleştirmek

🐍 **Python:** `{ ... }` bir **sözlüktür** (dictionary): `anahtar: değer` çiftlerinden oluşur. Burada her anahtar, künyede bir **sütun adı**; her değer, yukarıda tek tek hesapladığımız sütun adı → sayı eşlemelerinden biri. `pd.DataFrame(sözlük)` bunları **etiketlerine göre** yan yana dizer: aynı sütun adının (ör. `insu`) dört değeri aynı satıra gelir.

Komut birden çok satıra yayılmış: süslü parantez kapanana kadar Python devam ettiğini bilir. Hizalamak için konan boşluklar sonucu etkilemez.

📊 **Konu:** Sonuç tablosunda **satırlar**, `df`'in sütunlarıdır (öznitelikler); **sütunlar**, her öznitelik için bir kontroldür. Veri tablosu ile künye tablosu bu yüzden birbirinin "yan yatmış" hâlidir.

In [ ]:
kunye = pd.DataFrame({
    "dtype":      df.dtypes.astype(str),          # saklama tipi (Weka: Type)
    "eksik_oran": df.isna().mean().round(3),      # NaN oranı (Weka: Missing)
    "farkli":     df.nunique(),                   # farklı değer sayısı (Weka: Distinct)
    "sifir_oran": (sayisal == 0).mean().round(3), # 0 olan değerlerin oranı
})

🐍 **Python:** `len(df)` tablonun satır sayısıdır (768). `kunye["farkli"] == len(df)` her satır için True/False üretir. Eşittirin solundaki `kunye["yeni_ad"] = ...` tabloya **yeni bir sütun ekler**.

📊 **Konu:** Bir sütundaki farklı değer sayısı satır sayısına eşitse, her kaydın değeri farklıdır: hasta no, sipariş no gibi bir **kimlik** olabilir. Kimlik sütunu modele girerse model kayıtları ezberler; 3. haftada ele alıyoruz.

In [ ]:
kunye["kimlik_suphesi"] = kunye["farkli"] == len(df)

🐍 **Python:** Tabloyu yazdırır.

📊 **Konu:** Satırlar alfabetik sıradadır (`age`, `class`, `insu`, …), çünkü dört eşleme aynı etiketleri taşımıyordu (sıfır oranında `class` yok); pandas birleştirirken etiketleri sıralar. `class` satırında `sifir_oran` **NaN** çıkar: o hesap yalnız sayısal sütunlarda yapıldı.

In [ ]:
print(kunye)

#### f) Sayısal özet: `describe()`

🐍 **Python:** `describe()` her **sayısal sütun** için sekiz özet hesaplar ve bunları satır satır yazar. `.T` (*transpose*) tabloyu çevirir: öznitelikler satıra, özetler sütuna geçer, okumak kolaylaşır.

📊 **Konu:** Sütunlar: `count` eksik olmayan değer sayısı, `mean` ortalama, `std` standart sapma, `min` en küçük, `25%` birinci çeyrek (Q1), `50%` **medyan**, `75%` üçüncü çeyrek (Q3), `max` en büyük. `min`, Q1, medyan, Q3, `max` birlikte 2. haftadaki **beş sayı özetidir**. Weka panelinde medyan ve çeyrekler yoktur.

In [ ]:
df.describe().T.round(2)   # 25% = Q1, 50% = medyan, 75% = Q3

✅ **Sonucu okuyalım:** Artık `eksik_oran` sütunu gerçeği gösteriyor: `insu` %48,7, `skin` %29,6, `pres` %4,6, `mass` %1,4, `plas` %0,7 eksik. Bu sütunlarda `sifir_oran` 0'a indi; yalnız `preg`'de (%14,5) sıfırlar kaldı, çünkü 0 gebelik gerçek bir değerdir. `describe()`'da `count` sütunu da artık eksikleri yansıtıyor (ör. `insu` için 394) ve `insu`'nun ortalaması 79,8'den 155,5'e çıktı: sıfırlar ortalamayı aşağı çekiyormuş.

## 4. Eksik değerleri doldurmak: medyan ve gösterge sütunu

**Konu.** İşaretlenen eksikleri, sütunun **medyanıyla** dolduracağız. Neden medyan? `insu` sağa çarpık; ortalama uç değerlere çekilir, medyan çekilmez. Ayrıca her sütun için **"bu değer eksikti"** bilgisini 0/1 bir **gösterge sütunu** olarak saklayacağız: insülinin ölçülmemiş olması tek başına bir bilgi olabilir (belki doktor gerek görmediği için ölçülmedi).

Not: Doldurma da veriden bir şey **öğrenir** (medyanı). Bir modeli değerlendirirken medyan yalnız eğitim verisinden öğrenilmelidir; bölüm 8'deki sızıntı deneyi bunun nedenini gösteriyor. Burada yalnız doldurmanın etkisine bakıyoruz.

🐍 **Python:** scikit-learn'ün `impute` (doldurma) bölümünden `SimpleImputer` sınıfını açar.

📊 **Konu:** `SimpleImputer`, eksik değerleri basit bir kuralla (ortalama, medyan, en sık değer ya da sabit) doldurur.

In [ ]:
from sklearn.impute import SimpleImputer

🐍 **Python:** Bölüm 2'deki işaretleme, sunumda bu hücrenin de başında var ki hücre tek başına çalışsın. İkinci kez çalışması zararsızdır: artık değiştirilecek 0 kalmadı.

📊 **Konu:** Önce işaretle, sonra doldur.

In [ ]:
df[sutunlar] = df[sutunlar].replace(0, np.nan)          # 1) işaretle
print(df[sutunlar].isna().mean().round(3))             # eksik oranları

🐍 **Python:** `df["insu"]` tek bir sütun seçer (tek köşeli parantez). `.mean()` ortalaması, `.std()` standart sapması. pandas bunları hesaplarken **NaN'ları atlar**. `print`'e virgülle birden çok şey verilebilir; tırnak içindeki metin olduğu gibi yazılır.

📊 **Konu:** Doldurmadan **önce**, yalnız ölçülmüş 394 insülin değerinin ortalaması ve standart sapması.

In [ ]:
print("önce :", df["insu"].mean().round(1), df["insu"].std().round(1))

🔍 **Ara adım:** Doldurmada kullanılacak sayı: ölçülmüş insülin değerlerinin medyanı. `notna()` isna'nın tersidir (eksik değil mi?). 394 ölçülmüş, 374 eksik değer var; 374 hücrenin tamamı aynı sayıyla (125) doldurulacak.

In [ ]:
print("medyan:", df["insu"].median(), " ölçülmüş:", df["insu"].notna().sum(), " eksik:", df["insu"].isna().sum())

🐍 **Python:** **Sınıf ve nesne:** `SimpleImputer` bir kalıptır (sınıf); parantezle çağrılınca verilen ayarlarla bir **nesne** oluşur ve `doldurucu` adına atanır. Henüz hiçbir şey hesaplanmadı. `True`/`False` Python'un mantıksal değerleridir.

📊 **Konu:** `strategy="median"`: her sütunu kendi medyanıyla doldur. `add_indicator=True`: eksik değeri olan her sütun için bir 0/1 gösterge sütunu ekle. (Weka'nın ReplaceMissingValues filtresi yalnız ortalama ve en sık değerle doldurur, gösterge eklemez.)

In [ ]:
doldurucu = SimpleImputer(strategy="median", add_indicator=True)   # 2) doldur

🐍 **Python:** `fit_transform` iki işi arka arkaya yapar: `fit` veriden **öğrenir** (her sütunun medyanı), `transform` veriyi **dönüştürür** (NaN'ları o medyanla değiştirir, gösterge sütunlarını ekler). Sonuç bir numpy **dizisidir**: sütun adları yoktur, sütunlara sıra numarasıyla ulaşılır.

📊 **Konu:** Yalnız beş sütun (`sutunlar`) dolduruluyor; `preg`, `pedi`, `age` ve `class` bu işleme girmiyor.

In [ ]:
X_dolu = doldurucu.fit_transform(df[sutunlar])

🔍 **Ara adım:** `statistics_`: `fit` sırasında öğrenilen beş medyan (sırayla plas, pres, skin, insu, mass). Sonu alt çizgiyle biten özellikler öğrenilmiş değerlerdir. İkinci satır ilk 5 kaydı gösterir: ilk beş sütun doldurulmuş değerler, son beş sütun göstergeler (1 = bu değer eksikti). Örneğin ilk kaydın insülini eksikti: 4. sütunda 125, 9. sütunda 1 var.

In [ ]:
print(doldurucu.statistics_)
print(X_dolu[:5].round(1))

🐍 **Python:** `X_dolu[:, 3]`: dizide **[satır, sütun]** seçimi. `:` "bütün satırlar" demektir; `3` dördüncü sütundur (Python saymaya 0'dan başlar: 0 plas, 1 pres, 2 skin, **3 insu**, 4 mass). Yani bu, doldurulmuş insülin sütunudur.

📊 **Konu:** Doldurmadan **sonra**, 768 değerin ortalaması ve standart sapması.

In [ ]:
print("sonra:", X_dolu[:, 3].mean().round(1), X_dolu[:, 3].std().round(1))

🐍 **Python:** `.shape` bir **demettir** (tuple): değiştirilemeyen, parantezli bir liste. Burada (satır, sütun).

📊 **Konu:** Beş doldurulmuş sütun ve beş gösterge sütunu (beş sütunun hepsinde eksik vardı): 10 sütun.

In [ ]:
print(X_dolu.shape)   # 5 sütun + eksiği olan sütunlar için gösterge sütunları

✅ **Sonucu okuyalım:** Önce: ortalama **155,5**, standart sapma **118,8**. Sonra: ortalama **140,7**, standart sapma **86,3**. Verinin neredeyse yarısına aynı değer (125) yazıldığı için dağılım ortaya doğru **daralır**: standart sapma belirgin biçimde düşer ve ortalama medyana doğru kayar. Doldurma veriyi "olduğundan daha kesin" gösterir; bu yüzden gösterge sütunu ve doldurma kararının rapora yazılması önemlidir.

(Küçük bir ayrıntı: pandas'ın `.std()`'u n − 1'e, numpy'nin `.std()`'u n'ye böler. Bu fark 768 kayıtta önemsizdir; buradaki düşüşün nedeni doldurmadır.)

## 5. Kimlik sütunları ve yinelenen kayıtlar

**Konu.** İki kontrol: (1) Her değeri tek olan bir sütun (farklı değer sayısı = satır sayısı) bir **kimlik** olabilir (hasta no, sipariş no). Kimlik modele girerse model kayıtları ezberler. Kimlik sütunu modelden çıkarılır ama **atılmaz**: aynı kişinin kayıtlarını eğitim ve teste ayrı düşürmemek için kullanılır. (2) Bire bir aynı satırlar (**yinelenen kayıtlar**) hem istatistikleri bozar hem de aynı kayıt eğitimde ve testte birlikte bulunursa sızıntıya yol açar.

🐍 **Python:** `nunique()` her sütundaki farklı değer sayısını verir; sonuç sütun adı → sayı eşlemesidir (*Series*).

📊 **Konu:** Weka'daki *Distinct* ve *Unique* bilgisinin karşılığı.

In [ ]:
farkli = df.nunique()

🔍 **Ara adım:** Ara adım: önce sayılar, sonra her sütun için "satır sayısına (768) eşit mi?" sorusunun True/False cevabı. Hepsi False.

In [ ]:
print(farkli)
print(farkli == len(df))

🐍 **Python:** **Mantıksal filtreleme:** köşeli parantez içine bir True/False dizisi konunca yalnız **True** olan öğeler seçilir.

📊 **Konu:** Yalnız kimlik şüphesi taşıyan sütunlar listelenir.

In [ ]:
print(farkli[farkli == len(df)])

🐍 **Python:** `duplicated()` her satır için "bu satırın aynısı daha önce geçti mi?" (True/False); `.sum()` True'ları sayar.

📊 **Konu:** Bire bir aynı satır sayısı.

In [ ]:
print("yinelenen satır:", df.duplicated().sum())

Sunumdaki hücrenin son iki satırı **yorum** olarak bırakılmıştır, çünkü bu veride kimlik sütunu yok. Kendi verinizde bir hasta numarası varsa `#` işaretlerini silerek kullanırsınız:

```python
# from sklearn.model_selection import GroupKFold
# cv = GroupKFold(n_splits=5); cross_val_score(model, X, y, cv=cv, groups=df["hasta_no"])
```

📊 **Konu:** `GroupKFold`, aynı gruba (ör. aynı hastaya) ait bütün kayıtları **aynı** kata koyar. Böylece bir hastanın bir ziyareti eğitimde, öbür ziyareti testte olmaz; model o hastayı "tanıyarak" yüksek skor alamaz.

✅ **Sonucu okuyalım:** Çıktı boş bir liste (`Series([], ...)`): kimlik şüphesi taşıyan sütun yok. Yinelenen satır **0**. Bu veri için iki kontrol de temiz; kendi verinizde mutlaka yapın.

## 6. Uzaklık ve benzerlik hesaplarını doğrulamak

**Konu.** 2. haftada elle yaptığımız uzaklık ve benzerlik hesaplarını kodla tekrar ediyoruz; sayılar elle bulduklarınızla aynı olmalı. Üç örnek var:

1. **Sayısal vektörler:** `b`, `a`'nın tam 10 katı. Öklid, Manhattan, Chebyshev uzaklıkları ve kosinüs benzerliği.
2. **İkili (asimetrik) veri:** iki hastanın 10 taramalık sonucu. Jaccard ve basit eşleşme.
3. **Ölçekleme:** yaş ve gelir. Ölçeklemeden önce ve sonra Öklid uzaklığı.

🐍 **Python:** numpy modülünü `np` takma adıyla açar.

📊 **Konu:** numpy dizileriyle vektör hesabı yapacağız.

In [ ]:
import numpy as np

🐍 **Python:** scipy kütüphanesinin uzaklık bölümünden beş işlev açılıyor. Virgülle birden çok ad aynı satırda açılabilir.

📊 **Konu:** * `euclidean`: Öklid (L2) uzaklığı, √Σ(xᵢ − yᵢ)²
* `cityblock`: Manhattan (L1) uzaklığı, Σ|xᵢ − yᵢ|
* `chebyshev`: Chebyshev uzaklığı, en büyük |xᵢ − yᵢ|
* `cosine`: **kosinüs uzaklığı** = 1 − kosinüs benzerliği
* `jaccard`: **Jaccard uzaklığı** = 1 − Jaccard benzerliği

Dikkat: son ikisi benzerlik değil **uzaklık** döndürür. Benzerliği bulmak için 1'den çıkaracağız. Belgesini okumadan kullanılan bir işlev, istediğinizin tam tersini ölçüyor olabilir.

In [ ]:
from scipy.spatial.distance import euclidean, cityblock, chebyshev, cosine, jaccard

### 1. Sayısal vektörler

🐍 **Python:** `[2, 3, 0, 1]` bir Python **listesi**; `np.array(...)` onu bir numpy **dizisine** çevirir. Fark önemlidir: `10 * dizi` her elemanı 10 ile çarpar; `10 * liste` ise listeyi 10 kez arka arkaya ekler. Sayısal hesap için bu yüzden numpy dizisi kullanılır. Noktalı virgül (`;`) iki komutu tek satıra yazmaya yarar.

📊 **Konu:** `b`, `a` ile **aynı yönde** ama 10 kat büyük bir vektör.

In [ ]:
a = np.array([2, 3, 0, 1]); b = 10 * a

🔍 **Ara adım:** İlk satır iki diziyi gösterir: `b = [20 30 0 10]`. İkinci satır listeyle çarpmanın farkını gösterir: liste iki kez arka arkaya eklenir, sayılar çarpılmaz.

In [ ]:
print(a, b)
print([2, 3, 0, 1] * 2)

🔍 **Ara adım:** Öklid, Manhattan ve Chebyshev'i formülden elle hesaplayalım. `a - b` elemanları tek tek çıkarır. `** 2` kare alır, `.sum()` toplar, `np.sqrt` karekök alır: Öklid. `np.abs` mutlak değer: toplamı Manhattan, en büyüğü (`.max()`) Chebyshev.

In [ ]:
fark = a - b
print(fark)
print(np.sqrt((fark ** 2).sum()), np.abs(fark).sum(), np.abs(fark).max())

🐍 **Python:** Dört hesap tek `print` içinde, virgülle ayrılmış. `round(x, 3)` üç basamağa yuvarlar.

📊 **Konu:** `1 - cosine(a, b)`: scipy kosinüs **uzaklığı** verdiği için 1'den çıkarıp **benzerliği** buluyoruz.

In [ ]:
print(round(euclidean(a, b), 3), cityblock(a, b), chebyshev(a, b), round(1 - cosine(a, b), 3))

✅ **Sonucu okuyalım:** Öklid **33,675**, Manhattan **54**, Chebyshev **27** (en büyük fark: 3 ile 30), kosinüs benzerliği **1,0**. Öklid'e göre iki vektör uzak, kosinüse göre **özdeş** (aynı yön). Hangisi doğru? Soruya bağlı: büyüklük önemliyse Öklid, yalnız oranlar önemliyse kosinüs. Ölçüt seçimi bir modelleme kararıdır.

### 2. İkili veri: Jaccard ve basit eşleşme

🐍 **Python:** `dtype=bool`: dizinin elemanları sayı değil, **True/False** olarak saklanır (1 → True, 0 → False).

📊 **Konu:** On taramalık bir panelde iki hasta: 1 = pozitif bulgu, 0 = bulgu yok. Varlık bilgi taşır, yokluk çoğu zaman taşımaz: **asimetrik ikili** veri.

In [ ]:
x = np.array([1,0,1,0,0,0,0,1,0,0], dtype=bool)
y = np.array([1,0,0,0,0,0,0,1,0,1], dtype=bool)

🔍 **Ara adım:** Dört sayıyı elle sayalım. `&` "ve", `~` "değil" demektir; her ikisi de elemanlar üzerinde tek tek çalışır. q: ikisinde de 1 (tarama 1 ve 8), r: yalnız x'te 1, s: yalnız y'de 1, t: ikisinde de 0.

In [ ]:
q = (x & y).sum();  r = (x & ~y).sum();  s = (~x & y).sum();  t = (~x & ~y).sum()
print("q =", q, " r =", r, " s =", s, " t =", t)

🐍 **Python:** `x == y` her tarama için "iki hastanın sonucu aynı mı?" sorusunun True/False cevabıdır; `.mean()` True'ların **oranıdır**. Yine `1 - jaccard(...)` ile uzaklıktan benzerliğe geçiyoruz.

📊 **Konu:** Jaccard = q / (q + r + s) = 2 / 4; t'yi (ikisinde de bulgu yok) saymaz. Basit eşleşme = (q + t) / 10 = 8 / 10; t'yi sayar.

In [ ]:
print("Jaccard:", round(1 - jaccard(x, y), 3), " Basit eşleşme:", (x == y).mean())

✅ **Sonucu okuyalım:** Jaccard **0,5**, basit eşleşme **0,8**. Basit eşleşme, iki hastanın da bulgusu olmayan 6 taramayı "benzerlik" saydığı için şişiyor. Tarama verisi asimetrik olduğundan Jaccard uygundur. Panel 200 taramalık olsaydı basit eşleşme 1'e yaklaşırdı, Jaccard değişmezdi.

### 3. Ölçekleme: yaş ve gelir

🐍 **Python:** Çoklu atama: sağdaki üç dizi sırayla `A`, `B`, `C` adlarına atanır. Her dizi bir kişi: [yaş, gelir (TL)].

📊 **Konu:** A ile B yaşça yakın (25 ve 30), C ise çok farklı (60). Gelirleri birbirine yakın.

In [ ]:
A, B, C = np.array([25, 40000]), np.array([30, 42000]), np.array([60, 41000])

🐍 **Python:** İki Öklid uzaklığı, bir basamağa yuvarlanmış.

📊 **Konu:** Ölçeklemeden önce.

In [ ]:
print(round(euclidean(A, B), 1), round(euclidean(A, C), 1))

✅ **Sonucu okuyalım:** d(A, B) ≈ **2000**, d(A, C) ≈ **1000,6**. Öklid'e göre A'ya en yakın kişi **C**! Gelir farkı binlerle, yaş farkı onlarla ölçüldüğü için uzaklığı neredeyse tamamen gelir belirliyor: yaş hesapta yok gibi.

🐍 **Python:** İki dizi: `alt` her özniteliğin en küçük değeri, `genis` her özniteliğin **aralık genişliği**, yani maks − min.

📊 **Konu:** Yaş 18–70 aralığında: alt = 18, genişlik = 70 − 18 = **52**. Gelir 15 000–200 000 TL aralığında: alt = 15 000, genişlik = 200 000 − 15 000 = **185 000**. Dikkat: `genis` maksimum değil, maksimum ile minimumun **farkıdır**; bu yüzden formülde ayrıca `- alt` yazılmaz.

In [ ]:
alt, genis = np.array([18, 15000]), np.array([52, 185000])

🐍 **Python:** `def` ile kendi **işlevimizi** tanımlıyoruz: adı `minmaks`, aldığı değer (parametre) `p`. İki noktadan sonraki **girintili** satır işlevin gövdesidir; `return` sonucu geri verir. Python'da girinti zorunludur, bloğu o belirler. İşlev tanımlanınca hemen çalışmaz; çağrıldığında çalışır.

`(p - alt) / genis` numpy dizileriyle **eleman eleman** yapılır: yaştan 18 çıkarılıp 52'ye, gelirden 15 000 çıkarılıp 185 000'e bölünür.

📊 **Konu:** **Min-maks ölçekleme:** (x − min) / (maks − min). Her öznitelik [0, 1] aralığına gelir: en küçük değer 0, en büyük değer 1 olur. Böylece yaş ve gelir uzaklığa eşit ağırlıkla katılır.

In [ ]:
def minmaks(p):                  # min-maks ölçekleme: (x − min) / (maks − min)
    return (p - alt) / genis

🔍 **Ara adım:** Ölçeklenmiş değerler. A = (7/52, 25 000/185 000) ≈ (0,135; 0,135). İkinci satır kontrol: en küçük değerler (18, 15 000) 0'a, en büyük değerler (70, 200 000) 1'e gidiyor; ölçekleme doğru.

In [ ]:
print(minmaks(A), minmaks(B), minmaks(C))
print(minmaks(np.array([18, 15000])), minmaks(np.array([70, 200000])))

🐍 **Python:** İç içe çağrı, içten dışa: önce iki kişi ölçeklenir, sonra aralarındaki Öklid uzaklığı hesaplanır, sonra yuvarlanır.

📊 **Konu:** Ölçeklemeden sonra.

In [ ]:
print(round(euclidean(minmaks(A), minmaks(B)), 3), round(euclidean(minmaks(A), minmaks(C)), 3))

✅ **Sonucu okuyalım:** Ölçekleme sonrası d(A, B) ≈ **0,097**, d(A, C) ≈ **0,673**. Artık A'ya en yakın kişi **B**: sıralama tersine döndü. Ölçekleme tarafsız değildir; "her özniteliği eşit sayıyorum" demektir ve bu bir karardır. 3. haftada bir uyarı ekleyeceğiz: ölçekleyicinin min ve maks değerleri yalnız **eğitim** verisinden öğrenilmelidir, yoksa sızıntı olur.

**Sunumdaki hâli** (tek hücre):

In [ ]:
import numpy as np
from scipy.spatial.distance import euclidean, cityblock, chebyshev, cosine, jaccard

a = np.array([2, 3, 0, 1]); b = 10 * a
print(round(euclidean(a, b), 3), cityblock(a, b), chebyshev(a, b), round(1 - cosine(a, b), 3))

x = np.array([1,0,1,0,0,0,0,1,0,0], dtype=bool)
y = np.array([1,0,0,0,0,0,0,1,0,1], dtype=bool)
print("Jaccard:", round(1 - jaccard(x, y), 3), " Basit eşleşme:", (x == y).mean())

A, B, C = np.array([25, 40000]), np.array([30, 42000]), np.array([60, 41000])
print(round(euclidean(A, B), 1), round(euclidean(A, C), 1))
alt, genis = np.array([18, 15000]), np.array([52, 185000])
def minmaks(p):                  # min-maks ölçekleme: (x − min) / (maks − min)
    return (p - alt) / genis

print(round(euclidean(minmaks(A), minmaks(B)), 3), round(euclidean(minmaks(A), minmaks(C)), 3))

## 7. Ayrıklaştırma ve log dönüşümü

**Konu.** **Ayrıklaştırma** (discretization), sayısal bir değişkeni aralıklara (kovalara) bölüp kategorik hâle getirir. İki yaygın yol:
* **Eşit genişlik** (`uniform`): aralıkların **genişliği** eşittir; kayıt sayıları farklı olabilir.
* **Eşit frekans** (`quantile`): her aralıkta yaklaşık **aynı sayıda kayıt** vardır; genişlikler farklı olur.

**Log dönüşümü**, sağa çarpık bir değişkenin (birkaç çok büyük değeri olan) uzun kuyruğunu toplar ve dağılımı simetriğe yaklaştırır.

🐍 **Python:** scikit-learn'ün `preprocessing` (ön işleme) bölümünden `KBinsDiscretizer` sınıfını açar.

📊 **Konu:** `KBinsDiscretizer`: sayısal sütunu K kovaya bölen ayrıklaştırıcı.

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

Döngünün **tek bir turunu** önce elle yapalım (`strateji = "uniform"`).

🔍 **Ara adım:** `n_bins=4`: 4 kova. `encode="ordinal"`: her kaydın kovası 0, 1, 2, 3 diye **sıra numarasıyla** yazılsın. `df[["age"]]` (**çift** köşeli parantez) tek sütunlu bir **tablodur**; `df["age"]` (tek) ise bir **sütundur**. scikit-learn tablo beklediği için çift parantez gerekir. Son iki satır ilk beş kaydın yaşını ve düştüğü kovayı gösterir: örneğin 50 yaş **1** numaralı kovaya (36 ile 51 arası) düşüyor; kova numaraları 0'dan başlar.

In [ ]:
kb = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="uniform")
yas_kova = kb.fit_transform(df[["age"]])
print(df["age"].head().tolist())
print(yas_kova[:5, 0])

🔍 **Ara adım:** `bin_edges_`: `fit` sırasında öğrenilen **kova sınırları**. Her sütun için ayrı bir dizi tutulur; tek sütunumuz olduğu için `[0]` ile ilkini alıyoruz. 5 sınır 4 kova tanımlar.

In [ ]:
print(kb.bin_edges_)
print(kb.bin_edges_[0].round(1))

🔍 **Ara adım:** `yas_kova[:, 0]` kova numaraları sütunu (ondalıklı sayı olarak saklanır); `.astype(int)` tam sayıya çevirir. `np.bincount` her numaradan kaç tane olduğunu sayar: 0 numaralı kovada kaç kayıt, 1 numaralıda kaç kayıt, …

In [ ]:
print(yas_kova[:, 0].astype(int)[:10])
print(np.bincount(yas_kova[:, 0].astype(int)))

🐍 **Python:** **for döngüsü:** listedeki her öğe için girintili üç satırı bir kez çalıştırır; `strateji` önce `"uniform"`, sonra `"quantile"` olur. Her turda yeni bir ayrıklaştırıcı kurulur, yaşlar kovalara bölünür ve sınırlar ile kova başına kayıt sayısı yazılır.

📊 **Konu:** `uniform` Weka'daki eşit genişliğe, `quantile` eşit frekansa (Weka'da `useEqualFrequency`) karşılık gelir.

In [ ]:
for strateji in ["uniform", "quantile"]:          # eşit genişlik / eşit frekans
    kb = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy=strateji)
    yas_kova = kb.fit_transform(df[["age"]])
    print(strateji, kb.bin_edges_[0].round(1), np.bincount(yas_kova[:, 0].astype(int)))

✅ **Sonucu okuyalım:** **Eşit genişlik:** sınırlar 21, 36, 51, 66, 81 (her aralık 15 yıl); kayıt sayıları **498, 189, 68, 13**. Yaş sağa çarpık olduğu için kayıtların çoğu ilk kovada toplanır, son kovada 13 kişi kalır.

**Eşit frekans:** sınırlar 21, 24, 29, 41, 81; kayıt sayıları **173, 194, 207, 194**: yaklaşık eşit (her biri yaklaşık 768 / 4 = 192). Tam eşit olmamasının nedeni yaşların tam sayı olması: aynı yaştaki herkes aynı kovaya düşmek zorunda. Aralıkların genişliği ise çok farklı (3 yıldan 40 yıla).

(Colab'daki scikit-learn sürümüne göre `quantile` için bir uyarı mesajı görebilirsiniz; sonucu etkilemez.)

🐍 **Python:** `df["yeni_ad"] = ...` tabloya yeni bir sütun ekler. `np.log1p` bütün sütuna **tek seferde**, eleman eleman uygulanır (döngü gerekmez).

📊 **Konu:** `np.log1p(x)` = log(1 + x): x = 0 olsa bile tanımlıdır. NaN'lar NaN olarak kalır.

In [ ]:
df["insu_log"] = np.log1p(df["insu"])

🔍 **Ara adım:** İki sütun yan yana: 94 → log(95) ≈ 4,55; 168 → log(169) ≈ 5,13. Büyük değerler çok daha fazla "sıkıştırılır": 846 → 6,74.

In [ ]:
print(df[["insu", "insu_log"]].head(6))

🐍 **Python:** `df[["insu", "insu_log"]]` iki sütunluk bir tablo; `.skew()` **her sütunun** çarpıklığını hesaplar (NaN'ları atlayarak).

📊 **Konu:** **Çarpıklık** (skewness): 0 simetrik dağılım; pozitif değer sağa (büyük değerlere doğru) uzun kuyruk.

In [ ]:
print(df[["insu", "insu_log"]].skew().round(2))   # çarpıklık azalır

✅ **Sonucu okuyalım:** `insu` çarpıklığı **2,17** (belirgin sağa çarpık), `insu_log` **−0,09** (neredeyse simetrik). Uzaklığa dayalı yöntemler ve doğrusal modeller simetrik dağılımlarla daha iyi çalışır; ağaç tabanlı modeller için log dönüşümü çoğu zaman gerekmez (sıralamayı değiştirmez).

## 8. Sızıntı deneyi: öznitelik seçimi nerede yapılmalı?

**Konu.** **Sızıntı** (leakage): test verisinin bilgisinin, bir şekilde modelin eğitimine karışmasıdır. Sonuç olduğundan iyi görünür, model gerçekte o kadar iyi değildir. En sinsi biçimi şudur: öznitelik seçimi (ya da ölçekleme, doldurma) **bütün veride** yapılır, sonra çapraz doğrulama uygulanır.

**Deney.** Tamamen rastgele bir veri üretiyoruz: 100 kayıt, **5000 rastgele öznitelik**, **rastgele** 0/1 etiket. Öğrenilecek hiçbir şey yok; dürüst bir değerlendirme yaklaşık **%50** bulmalı. Aynı veride iki kurulumu karşılaştıracağız.

**Çapraz doğrulama** (7. haftada ayrıntılı): veri 5 parçaya (kat) bölünür; her seferinde bir kat test, kalan dördü eğitim olur. 5 skorun ortalaması raporlanır.

🐍 **Python:** Dört satır, beş araç (`numpy` yukarıda açılmıştı).

📊 **Konu:** * `SelectKBest`: öznitelikleri bir puana göre sıralayıp en iyi **k** tanesini seçer.
* `f_classif`: bu puan; her özniteliğin sınıflar arasında ne kadar ayrıştığını ölçen ANOVA F istatistiği. Seçim **etiketlere bakarak** yapılır.
* `LogisticRegression`: lojistik regresyon sınıflandırıcısı.
* `cross_val_score`: çapraz doğrulama yapıp her katın skorunu döndürür.
* `StratifiedKFold`: sınıf oranlarını koruyan kat bölücüsü.
* `Pipeline`: birden çok adımı (seçim → model) tek bir **hat** hâline getirir.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

🐍 **Python:** `rng` bir **rastgele sayı üretecidir**; 42 tohumdur: her çalıştırmada aynı "rastgele" sayılar gelir, sonuç tekrarlanabilir.

In [ ]:
rng = np.random.default_rng(42)

🐍 **Python:** `rng.normal` standart normal dağılımdan sayı üretir. `size=(100, 5000)` bir demettir: 100 satır, 5000 sütun.

📊 **Konu:** Öznitelik matrisi: **gürültüden** başka bir şey yok. Dikkat: 2. bölümdeki `df` ile ilgisi yok; `X` ve `y` adlarını yeniden kullanıyoruz.

In [ ]:
X = rng.normal(size=(100, 5000))   # 5000 tamamen rastgele öznitelik

🐍 **Python:** `rng.integers(0, 2, 100)`: 0 ile 2 arasında (2 dâhil değil) 100 tam sayı, yani 0 ya da 1.

📊 **Konu:** Etiketler özniteliklerle **ilişkisiz**.

In [ ]:
y = rng.integers(0, 2, 100)        # rastgele etiket: öğrenilecek bir şey yok

🔍 **Ara adım:** Boyutlar ve etiket dağılımı: 100 × 5000 matris, 100 etiket; 0 ve 1'lerin sayısı yaklaşık eşit.

In [ ]:
print(X.shape, y.shape, np.bincount(y))

🐍 **Python:** Bir kat bölücü nesne: 5 kat, bölmeden önce karıştır, tohum 42.

📊 **Konu:** İki kurulumda da **aynı** katlar kullanılsın diye bir kez tanımlıyoruz.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)

### Yanlış kurulum: önce seç, sonra çapraz doğrula

🐍 **Python:** Seçici nesnesi oluşturulur ve hemen `fit_transform` çağrılır: `fit` 5000 özniteliğin puanını **bütün 100 kayıt ve etiketleriyle** hesaplar, `transform` en iyi 20 sütunu döndürür.

📊 **Konu:** **Hata burada:** seçim, ileride test katında olacak kayıtların etiketlerine de bakıyor. 5000 rastgele sütunun içinden, **tesadüfen** bu 100 etiketle en çok örtüşen 20'si seçiliyor.

In [ ]:
Xs = SelectKBest(f_classif, k=20).fit_transform(X, y)

🔍 **Ara adım:** Artık 100 × 20: seçilmiş 20 öznitelik.

In [ ]:
print(Xs.shape)

🐍 **Python:** `cross_val_score(model, X, y, cv=...)` her kat için modeli eğitip test eder ve 5 skorluk bir dizi döndürür. `max_iter=1000`: modelin öğrenme adım sayısı üst sınırı (yakınsama uyarısı çıkmasın diye).

📊 **Konu:** Çapraz doğrulama yalnız **modeli** katlara göre eğitiyor; seçim ise zaten bütün veriyle yapılmıştı.

In [ ]:
a = cross_val_score(LogisticRegression(max_iter=1000), Xs, y, cv=cv)

🔍 **Ara adım:** Beş katın ayrı ayrı doğrulukları.

In [ ]:
print(a.round(2))

### Doğru kurulum: seçim hattın içinde

🐍 **Python:** `Pipeline`'a **(ad, nesne)** demetlerinden oluşan bir liste verilir; adımlar sırayla çalışır. Adlar (`"sec"`, `"model"`) bizim seçtiğimiz etiketlerdir.

📊 **Konu:** Hat bir bütün olarak eğitilir: `fit` çağrılınca önce seçici **yalnız o anki eğitim verisiyle** öğrenir, sonra model seçilen 20 öznitelikle eğitilir.

In [ ]:
hat = Pipeline([("sec", SelectKBest(f_classif, k=20)),
                ("model", LogisticRegression(max_iter=1000))])

🐍 **Python:** Bu kez `cross_val_score`'a modelin yerine **hattın tamamı** ve 5000 sütunluk **ham** `X` verilir.

📊 **Konu:** Her katta: seçim dört eğitim katıyla yapılır, test katı seçime hiç karışmaz. Her katta farklı 20 öznitelik seçilebilir; olması gereken de budur.

In [ ]:
b = cross_val_score(hat, X, y, cv=cv)

🔍 **Ara adım:** Beş katın doğrulukları.

In [ ]:
print(b.round(2))

🐍 **Python:** **f-string:** `f"..."` içinde süslü parantezdeki ifade hesaplanıp metne yazılır; `:.2f` iki ondalık basamak demektir. `a.mean()` 5 skorun ortalaması, `a.std()` standart sapması.

In [ ]:
print(f"Yanlış: {a.mean():.2f} ± {a.std():.2f}")
print(f"Doğru:  {b.mean():.2f} ± {b.std():.2f}")

**Sunumdaki hâli** (tek hücre):

In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

rng = np.random.default_rng(42)
X = rng.normal(size=(100, 5000))   # 5000 tamamen rastgele öznitelik
y = rng.integers(0, 2, 100)        # rastgele etiket: öğrenilecek bir şey yok
cv = StratifiedKFold(5, shuffle=True, random_state=42)

# YANLIŞ: öznitelik seçimi bütün veride, sonra çapraz doğrulama
Xs = SelectKBest(f_classif, k=20).fit_transform(X, y)
a = cross_val_score(LogisticRegression(max_iter=1000), Xs, y, cv=cv)

# DOĞRU: seçim hattın içinde, her katta yalnız eğitim verisiyle
hat = Pipeline([("sec", SelectKBest(f_classif, k=20)),
                ("model", LogisticRegression(max_iter=1000))])
b = cross_val_score(hat, X, y, cv=cv)
print(f"Yanlış: {a.mean():.2f} ± {a.std():.2f}")
print(f"Doğru:  {b.mean():.2f} ± {b.std():.2f}")

✅ **Sonucu okuyalım:** Yanlış kurulum **0,88 ± 0,08**, doğru kurulum **0,53 ± 0,12**. Veri tamamen rastgele olduğu için doğru cevap %50 civarıdır; doğru kurulum bunu buluyor. Yanlış kurulum %88 buldu, çünkü test kayıtları seçimi zaten etkilemişti: model onları "tanıyordu".

**Ders:** Kod hatasız çalıştı ve %88 verdi. Sızıntı ancak kurulumu sorgulayınca görünür. Öznitelik seçimi, ölçekleme ve doldurma gibi veriden **öğrenen** her adım hattın (`Pipeline`) içinde olmalıdır. Bu deneyi kendi verinizde (ölçekleme ve doldurmayla) yapmak Proje 1'in 1. bölümünün parçasıdır.